# Hector B0: founder-run BF16 evaluation
Run only reviewed Phase 01b code. Select the exact published commit accepted for this run; the Phase 01 starting commit has no harness. Stop at the first failure. This notebook has not been executed by Codex.

Accept Appendix A labels and authorize GPU use and the approximately 60 GB checkpoint download separately before changing the two authorization flags. There is no precision, model-class, backend or placement fallback. Mounting Drive grants access to the whole Drive; only reviewed code should run here. Use a fresh Colab VM for each attempt. Never resume or pool runs.


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

RUN_COMMIT = input("Exact reviewed Phase 01b repository commit (40 hex): ").strip()
if not re.fullmatch(r"[0-9a-f]{40}", RUN_COMMIT):
    raise RuntimeError("An exact reviewed commit SHA is required")
REPO = Path("/content/hector-evidence-agent")
if REPO.exists():
    raise RuntimeError("Preserve any earlier run; use a fresh Colab VM")
subprocess.run(["git", "clone", "--no-checkout", "https://github.com/TenzyZ/hector-evidence-agent.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", RUN_COMMIT], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from hector.b0_colab import check_repository
check_repository(REPO, RUN_COMMIT)
os.environ["HF_HOME"] = "/content/hf"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
Path("/content/hf").mkdir(exist_ok=True)


## Install cell (founder only)
Exact pins verified against PyPI on 2026-10-28: Transformers 5.20.2 released 2026-10-19, Accelerate 1.15.0 released 2026-09-09; both Apache 2.0. Sources: https://pypi.org/pypi/transformers/5.20.2/json and https://pypi.org/pypi/accelerate/1.15.0/json. Preserve Colab's existing PyTorch build with an exact constraint. Nothing installs torch explicitly. Any dependency conflict stops execution.


In [ ]:
import importlib.metadata
import tempfile

TORCH_BUILD = importlib.metadata.version("torch")
with tempfile.TemporaryDirectory(prefix="hector-pins-", dir="/content") as temporary:
    constraint = Path(temporary) / "constraints.txt"
    constraint.write_text("torch==" + TORCH_BUILD + "\n", encoding="utf-8")
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed",
                    "--constraint", str(constraint), "transformers==5.20.2", "accelerate==1.15.0"], check=True)
if importlib.metadata.version("torch") != TORCH_BUILD:
    raise RuntimeError("Colab PyTorch build changed; stop")


## Run identity and preservation
The following flags are founder decisions, not implementation acceptance. Enter the before-run compute-unit balance below when available. After generation, the driver asks for the then-observable balance before finalizing run.json and scoring; blank means unobserved. These balances are founder-entered measurements, separate from model telemetry. Checkpoint files remain in `/content/hf`, never Drive or Git. Each completed item and stage is mirrored to `MyDrive/hector-b0/<run_id>/`. A disconnected run remains `started`; start a new VM and run ID instead of resuming it.


In [ ]:
from google.colab import drive
from hector.b0_colab import B0Run, prompt_compute_units_after

drive.mount("/content/drive")
LABELS_ACCEPTED = False  # Founder changes only after accepting Appendix A labels.
DOWNLOAD_AUTHORIZED = False  # Founder changes only after authorizing GPU use and weight download.
COMPUTE_UNITS_BEFORE = None
run = B0Run(RUN_COMMIT, drive_root="/content/drive/MyDrive/hector-b0",
            labels_accepted=LABELS_ACCEPTED, compute_units_before=COMPUTE_UNITS_BEFORE)
print(run.path)


## B0-0: current environment
Requires clean canonical checkout, Python >=3.12, exact pins and VM-local cache. Package versions, PyTorch/CUDA build and repository identity are recorded. Prior sessions provide no stage credit.


In [ ]:
run.environment()


## B0-1: tokenizer and all prompts
Metadata and tokenizer files only. Model revision is fixed in `hector/b0_colab.py`. Sixteen rendered prompt hashes and token counts are recorded; no checkpoint is loaded.


In [ ]:
run.tokenizer_stage()


## P: resource gate (stop on any failure)
Checks CUDA/BF16, at least 72 GiB free VRAM, at least 24 GiB available host RAM (WEIGHTWRIGHT heuristic, not a Meta guarantee), disk for actual safetensors metadata plus 20 GiB, Python, pins, and anonymous model metadata access. No checkpoint download occurs on failure. Resources are checked again immediately before loading.


In [ ]:
run.preflight()


## B0-2, B0-3 and B0-4: explicitly authorized execution
Loads the pinned BF16 checkpoint with native `AutoModelForMultimodalLM` onto cuda:0; successful load evidence records the actual resolved class. The tokenizer-based text path is unchanged. Every parameter's placement and every floating dtype are checked. Then q01 must produce a genuine final message and valid JSON before the separate sixteen-item run begins. No retries or fallback. Two consecutive OOMs abort. Any unexpected model-loading class requirement stops the run. The model is released on success or failure.


In [ ]:
try:
    run.load(download_authorized=DOWNLOAD_AUTHORIZED)
    run.smoke()
    run.evaluate(compute_units_after=prompt_compute_units_after)
finally:
    run.release()


## Founder handoff
The after-run balance is collected after generation and before finalization; do not edit finalized run.json or scoring evidence. Copy the entire preserved run directory to the laptop, including stages.json, run.json and items.jsonl. Run `python -m hector.pilot_b0 score <run_dir>`: identical evidence is verified without rewriting; any mismatch stops with a non-zero exit and preserves the existing artifacts. A scoring failure aborts finalization. Never delete prior evidence to bypass a mismatch; retain it for founder review.

Review `review.csv` and `REPORT.md` against `summary.json`. Human labels remain separate from automatic scores. The report lists other run directories present on the VM; also disclose any other preserved Drive runs, including interrupted and aborted ones. Only after founder review copy permitted artifacts into `docs/eval/pilot-b0/`. A smoke response alone is never a baseline. Independent Opus acceptance, Git delivery and final G0 acceptance remain separate gates.
